<!-- nav -->
[← 2 · Choosing what to apply: candidates from a frame stream](02_Choosing_What_To_Apply.ipynb) · [Home](../../README.md) · [Equivalence →](../../docs/EQUIVALENCE.md)

# 3 · Running again: reruns and diffs, queries over the certificates

**The problem.** The shelf changes: an item is added, another removed. Run again, the rewrite should make the same
decisions where the same transform applies to the same frame, and ask only about what is new. And a reviewer wants to
know what changed between two runs: what was added, what was removed, what was decided differently.

mbse-transforms has no call for either. A transform certificate records its transform, its parameter assignments and
each sub-certificate's frame ("ofc those parameters must be recorded in the transform certificate"), and certificates
are mbse-schemas data. So a rerun's reuse is a query over the certificates' store (3.2), and a diff the same query
over two histories, with mbse-schemas' `Comparison` (3.3). Calls of mbse-transforms' own for them would be bespoke
code reading a model, which the user ruled out: "No model should need bespoke schema-aware code to read or write. Use
mbse-schemas built-in serialization support."

The shelf, its prototype and `label` are case study 1's; `run` is a session as in case study 2, with each frame's
case decided by a function you give it.

In [1]:
import { Builders, Comparison, Plain, Proxies, References } from "@mbse/schemas/Framework";
import { Transforms } from "@mbse/transforms";

import { assignments_of, bear, frames, items, model, resolve, shown } from "./toolkit.js";

/** Labels the sync's item with its name, in the case its parameter assignment gives. */
function label(source: any, sync: any, target: any, assignments: Transforms.Assignments): void {
  const item = sync.i;
  const name = String(item.name);
  (Proxies.store_of(item) as any).shelf.Item(item, (i: any) => i.label(assignments.case === "upper" ? name.toUpperCase() : name.toLowerCase()));
}

const transforms: any = model(); // the transforms' store
transforms.transaction(Transforms.install);
transforms.transaction((t: any) => Transforms.define(t, "Label", "label", t.shelf.Unlabelled));
const procedures = new Transforms.Procedures();
procedures.register("label", label);
const certs: any = new Builders.Store(); // the certificates' store
certs.transaction(Transforms.install);

type Decide = (frame: any) => [string, string];

/** Your run over the data's frames of shelf.Unlabelled: for each frame, `decide(frame)` gives its case and who
 * decided it; the candidate of Label with that case is applied, its frame the sync, and certified into the history
 * named `history`. */
function run(data: any, history: string, decide: Decide): void {
  const found = frames(data, "shelf.Unlabelled");
  transforms.transaction(() => data.transaction(() => {
    const Label = transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform;
    for (const candidates of new Transforms.Session([Label], found.map((f) => bear(data, "shelf.Unlabelled", f)))) {
      const [caseOf, by] = decide(candidates[0]!.frame);
      const [taken] = candidates.filter((c) => c.assignments.case === caseOf);
      Transforms.apply(Label, taken!.assignments, procedures, { sync: taken!.frame });
      Transforms.certify(certs, Label, taken!.assignments, history, by, { sync: taken!.frame });
    }
  }));
}

/** You deciding each item's case yourself, by its name. */
function chosen(choices: Record<string, string>): Decide {
  return (frame) => [choices[String(frame.i.name)]!, "caller"];
}

/** The key on the shelf of the item a sub-certificate's frame binds, read from the frame's plain data. */
function key_of(frame: any): string {
  const found: any[] = [];
  frame.as_object((o: any) => {
    found.push(Plain.ToPlain(certs)(o.as_schema(), o));
    return o;
  });
  return found[0].get("binds").get("i").get("path").get("next").get("entry").get("properties").get("key");
}

/** A transform certificate, read within a transaction: the key of the item its frame binds, and its parameter
 * assignments. */
function told(certificate: any): string {
  return `${key_of([...certificate.frames][0].frame.frame)} ${shown(assignments_of(certificate))}`;
}

/** Prints each transform certificate of the history named `name`, and who decided it. */
function history_of(name: string): void {
  certs.transaction(() => {
    const history = certs.singleton("Transforms.TheHistories").histories.at(name).history;
    const decisions = [...history.decisions];
    for (const [k, entry] of [...history.steps].entries()) console.log(`${told(entry.step)}, decided by ${decisions[k].by}`);
  });
}

<a id="T03.1"></a>
## 03.1 · A first run, recorded

A shelf of bolt and nut, each case decided by you, certified into the history `first`:

In [2]:
const first = model("bolt", "nut");
run(first, "first", chosen({ bolt: "upper", nut: "lower" }));
history_of("first");

bolt case=upper, decided by caller
nut case=lower, decided by caller


<a id="T03.2"></a>
## 03.2 · A rerun reuses what the certificates recorded

To reuse a decision, query the certificates' store for the transform certificate, in the earlier history, of the
same transform on the same frame: a `map` whose spec names the history, its `steps`' transform certificate whose
`transform` equals the transform's object reference, and its `frames`' sub-certificate whose `frame` equals the frame.
A spec matches a value by equality of the whole value, so the transform's reference and the frame's bindings are made
values of the certificates' store first, written as plain data and read back (`carried`). A frame binds by object
reference, so a shelf remade with the same owners and keys is queried by its own frames.

Where the query finds a certificate, read the parameter assignments it recorded, within a transaction, and apply and
certify with them, decided "reused"; where it finds none, decide as for a new frame.

In [3]:
/** The object reference of `x` (an object, or a frame's bindings), made in the store `own`, as a value of the
 * certificates' store: written as plain data and read back. */
function carried(own: any, x: unknown): any {
  const reference = References.of(own, x);
  return Plain.FromPlain(certs)(certs.registered(reference.as_schema().name),
    Plain.ToPlain(own)(reference.as_schema(), reference));
}

/** The query: the frames binding each transform certificate `c`, in the history named `history`, of the transform
 * `transform`, and its sub-certificate `f` holding the frame `frame`. */
function recorded(history: string, transform: any, frame: any): Map<string, any>[] {
  const T = certs.Transforms;
  return certs.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ d: T.Histories, h: T.HistoryCertificate, c: T.TransformCertificate, f: T.FrameCertificate })
    .spec([
      (_: any) => _.d.histories({ [history]: (e: any) => e.history(_.h) }),
      (_: any) => _.h.steps([(e: any) => e.step(_.c)]),
      (_: any) => _.c.transform(transform).frames([(e: any) => e.frame(_.f)]),
      (_: any) => _.f.frame(frame),
    ])]))]));
}

const label_ref = transforms.transaction(() =>
  carried(transforms, transforms.singleton("Transforms.TheCatalog").transforms.at("Label").transform));

/** Deciding as the history named `history` recorded, for a frame of `data`'s: the parameter assignments the query
 * finds, decided "reused"; where it finds none, `otherwise`'s decision. */
function reusing(history: string, data: any, otherwise: Decide): Decide {
  return (frame) => {
    const found = recorded(history, label_ref, carried(data, new Map([["i", frame.i]])));
    if (found.length === 0) return otherwise(frame); // a frame the earlier history never saw
    const [given] = found;
    return certs.transaction(() => // the certificate the frame given back binds, read within a transaction
      [String(assignments_of(resolve(certs, given!.get("c"))).case), "reused"]);
  };
}

const again = model("bolt", "nut", "washer"); // remade, with an item the first run never saw
run(again, "second", reusing("first", again, chosen({ washer: "upper" })));
history_of("second");
items(again);

bolt case=upper, decided by reused
nut case=lower, decided by reused
washer case=upper, decided by caller
bolt BOLT
nut nut
washer WASHER


<a id="T03.3"></a>
## 03.3 · A diff: two histories compared

A diff lists each history's certificates by the same kind of query, looks each up in the other history by the query
of 3.2, and compares each pair it finds with mbse-schemas' `Comparison.OfObject`: a certificate with none in the other
history was removed (an orphan) or added, and a pair not equal recorded other parameter assignments. Each history's
certificates are listed with their transform and frame, as values of the certificates' store, to query the other by.
In TypeScript, such a value is made after the visitor's callback returns, from the plain data read within it: one made
within the callback lives by that transaction ([Equivalence, Gaps found](../../docs/EQUIVALENCE.md#gaps-found)).

In [4]:
/** The query a diff starts from: the frames binding each transform certificate `c` in the history named `history`
 * and each of its sub-certificates `f`. */
function certified(history: string): Map<string, any>[] {
  const T = certs.Transforms;
  return certs.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
    .symbols({ d: T.Histories, h: T.HistoryCertificate, c: T.TransformCertificate, f: T.FrameCertificate })
    .spec([
      (_: any) => _.d.histories({ [history]: (e: any) => e.history(_.h) }),
      (_: any) => _.h.steps([(e: any) => e.step(_.c)]),
      (_: any) => _.c.frames([(e: any) => e.frame(_.f)]),
    ])]))]));
}

/** The value object a certificate's property holds, as a value of the certificates' store: written as plain data
 * within the callback, and read back after it returns. */
function held(member: any): any {
  const found: any[] = [];
  member.as_object((o: any) => {
    found.push([o.as_schema(), Plain.ToPlain(certs)(o.as_schema(), o)]);
    return o;
  });
  return Plain.FromPlain(certs)(found[0][0], found[0][1]);
}

/** Whether two transform certificates are equal as values (mbse-schemas' `Comparison`): their transform, prototype
 * and parameter assignments. */
function same(a: any, b: any): boolean {
  return new Comparison.OfObject(a.as_schema(), a).compare(new Comparison.OfObject(b.as_schema(), b)) === 0;
}

/** What the history named `b` removes from the one named `a`, what it adds, and what it changes, each by object
 * reference. */
function diff(a: string, b: string): [any[], any[], any[][]] {
  const listed = new Map<string, [any, any, any][]>();
  for (const name of [a, b]) {
    const found = certified(name);
    listed.set(name, certs.transaction(() => found.map((given): [any, any, any] => [given.get("c"),
      held(resolve(certs, given.get("c")).transform), held(resolve(certs, given.get("f")).frame)])));
  }
  const removed = listed.get(a)!.filter(([, transform, frame]) => recorded(b, transform, frame).length === 0).map(([c]) => c);
  const added = listed.get(b)!.filter(([, transform, frame]) => recorded(a, transform, frame).length === 0).map(([c]) => c);
  const pairs = listed.get(a)!.flatMap(([c, transform, frame]) => recorded(b, transform, frame).map((given) => [c, given.get("c")]));
  const changed = certs.transaction(() => pairs.filter(([c, d]) => !same(resolve(certs, c), resolve(certs, d))));
  return [removed, added, changed];
}

const [before, after] = [model("bolt", "nut", "washer"), model("bolt", "nut", "screw")]; // washer removed, screw added
run(before, "before", chosen({ bolt: "upper", nut: "upper", washer: "lower" }));
run(after, "after", chosen({ bolt: "upper", nut: "lower", screw: "lower" })); // nut decided otherwise
const [removed, added, changed] = diff("before", "after");
certs.transaction(() => {
  console.log(`removed: ${removed.map((c) => told(resolve(certs, c))).join("; ")}`);
  console.log(`added: ${added.map((c) => told(resolve(certs, c))).join("; ")}`);
  console.log(`changed: ${changed.map(([c, d]) => `${told(resolve(certs, c))} -> ${told(resolve(certs, d))}`).join("; ")}`);
});
console.log(`a rerun that reused, against its first run: [${diff("first", "second").map((found) => found.length).join(", ")}]`);

removed: washer case=lower
added: screw case=lower
changed: nut case=upper -> nut case=lower
a rerun that reused, against its first run: [0, 1, 0]


Bolt's certificates are equal in both histories, so it is in neither list; the rerun of 3.2 added only washer,
and changed nothing. Reruns and diffs need nothing more of mbse-transforms than the certificates it writes.

<!-- nav -->
[← 2 · Choosing what to apply: candidates from a frame stream](02_Choosing_What_To_Apply.ipynb) · [Home](../../README.md) · [Equivalence →](../../docs/EQUIVALENCE.md)